In [14]:
from pynq.overlays.base import BaseOverlay
import time
import multiprocessing as mp
import socket
import asyncio
base = BaseOverlay("base.bit")

In [16]:
%%microblaze base.PMODA

#include "gpio.h"

gpio pin_out = gpio_open(0); // default to pin 0

void select_pin(unsigned int pin){
    pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
}

//Function to turn on/off a selected pin of PMODA
int write_gpio(unsigned int pin, unsigned int val){
    gpio_write(pin_out, val);
    return 1;
}



In [18]:
import asyncio
btns = base.btns_gpio

# using await asyncio.sleep
async def tone_for_duration(freq, duration, pin):
    select_pin(pin)
    start=time.time()
    while start+duration>time.time():
        ret = write_gpio(pin, 1)
        await asyncio.sleep((1/freq*2))
        ret = write_gpio(pin, 0)
        await asyncio.sleep(1/(freq*2))

asyncio.run(tone_for_duration(5000, 1, 0))
print("done")

done


In [8]:
# PMOD PWM TEST

from pynq.lib.pmod.pmod_pwm import Pmod_PWM

pwm_man = Pmod_PWM(base.PMODA, 0)
usec_conv = 1e6
pwm_man.generate(int((1/2000)*usec_conv), 50)
time.sleep(1)
pwm_man.stop()


In [19]:
LIZ_IP = "192.168.8.207"
MY_IP = "192.168.8.225"
PORT_CLIENT = 5356
PORT_SERVER = 4376

LAPTOP_IP = "192.168.8.236"


def tone_for_duration(freq, duration, pin):
    select_pin(pin)
    start=time.time()
    while start+duration>time.time():
        ret = write_gpio(pin, 1)
        time.sleep(1/(freq*2))
        ret = write_gpio(pin, 0)
        time.sleep(1/(freq*2))

def wait_for_button(btns):
    while True:
        time.sleep(0.01)
        res = btns.read()
        if res & 0b0001:
            return 0
        if res & 0b0010:
            return 1
        if res & 0b0100:
            return 2
        if res & 0b1000:
            return 3

def client_proc(base):
    btns = base.btns_gpio
    while wait_for_button(btns)!=0:
        # wait for button 0 to start connection
        pass
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.connect((LIZ_IP, PORT_CLIENT))
    print("client connected!")
    time.sleep(0.5)
    while True:
        match wait_for_button(btns):
            case 1:
                # buzz
                sock.send(b"BUZZ!")
                time.sleep(0.5)
            case 3:
                sock.send(b"Close")
                break
            case _:
                print("wrong button")
    sock.close()
    print("client exit")

def server_proc(base):
    
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.bind((MY_IP, PORT_SERVER))
    try:
        sock.listen(2)
        con, adr = sock.accept()
        print("server received connection!")
        msg = con.recv(1024)
        while len(msg)>0:
            if msg.startswith(b"BUZZ!"):
                print("server got buzz")
                tone_for_duration(2000, 0.5, 0)
            elif msg.startswith(b"Close"):
                print("Close received")
            else:
                print(f"Invalid command {msg}")

            msg = con.recv(1024)
    finally:
        con.close()
        print("server exit")






In [24]:
# run 
client_p = mp.Process(target=client_proc, args=[base])
server_p = mp.Process(target=server_proc, args=[base])

client_p.start()
server_p.start()

client_p.join()
server_p.join()

client connected!
client exit
server received connection!
server got buzz
server got buzz
Close received
server exit


In [3]:
from pynq.lib.pmod.pmod_pwm import Pmod_PWM
base = BaseOverlay("base.bit")

In [12]:
# try with other buzz

def tone_for_duration(freq, duration, pin):
    pwm_man = Pmod_PWM(base.PMODA, pin)
    usec_conv = 1e6
    pwm_man.generate(int((1/freq)*usec_conv), 50)
    time.sleep(duration)
    pwm_man.stop()
        

client_p = mp.Process(target=client_proc, args=[base])
server_p = mp.Process(target=server_proc, args=[base])

client_p.start()
server_p.start()

client_p.join()
server_p.join()


Process Process-14:
Process Process-13:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/tmp/ipykernel_997/3598716167.py", line 60, in server_proc
    con, adr = sock.accept()
  File "/usr/lib/python3.10/socket.py", line 293, in accept
    fd, addr = self._accept()
KeyboardInterrupt
  File "/usr/lib/python3.10/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_997/3598716167.py", line 33, in client_proc
    while wait_for_button(btns)!=0:
  File "/tmp/ipykernel_997/3598716167.py", line 20, in wait_for_button
    time.sleep(0.01)
  File "/usr/lib/python3.10/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run


KeyboardInterrupt: 